# Complete Synthetic Series Demo

This notebook uses `TimeSeriesGenerator` from `foretools/tsgen/ts_gen.py` to build fully observed synthetic series with:

- piecewise trend
- weekly and monthly seasonality
- a slow cycle
- ARMA-style noise
- regime switching
- exogenous drivers
- calendar features

The plotting helper below shows the observed signal together with every tracked component in `meta["components"]`, plus the exogenous inputs, calendar encodings, and regime state path.

This example intentionally does **not** inject missing values, outliers, or multivariate factor mixing so the decomposition remains exact and the generated series stay complete.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from foretools.tsgen.ts_gen import TimeSeriesGenerator

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


In [ ]:
gen = TimeSeriesGenerator(random_state=42)

df, meta = gen.make(
    n_series=3,
    n_steps=365,
    freq="D",
    start="2024-01-01",
    trend={
        "type": "piecewise",
        "knots": [120, 250],
        "slopes": [0.08, -0.03, 0.05],
        "intercept": 18.0,
    },
    seasonality=[
        {"period": 7.0, "amplitude": 4.2, "harmonics": 2},
        {"period": 30.5, "amplitude": 1.8, "harmonics": 1},
    ],
    cycle={"period": 180.0, "amplitude": 2.4, "freq_drift": 0.08, "phase": 0.2},
    noise={"ar": [0.55], "ma": [0.2], "sigma": 0.9},
    regime={
        "n_states": 2,
        "p": [[0.96, 0.04], [0.08, 0.92]],
        "state_bias": [0.0, 3.2],
        "state_sigma_scale": [1.0, 1.35],
    },
    exog={
        "n_features": 2,
        "types": ["random_walk", "seasonal"],
        "beta": [0.22, -1.1],
    },
    add_calendar=True,
    return_components=True,
)

summary = df.groupby("series")["y"].agg(["count", "min", "max", "mean", "std"])
summary["complete"] = df.groupby("series")["y"].apply(lambda s: bool(s.notna().all()))
summary


In [ ]:
def plot_full_series_info(df: pd.DataFrame, meta: dict, series_id: int) -> None:
    d = df[df["series"] == series_id].copy()
    time = pd.to_datetime(d["time"])
    comps = meta["components"]
    params = meta["params"]
    exog_cols = meta["exog_cols"]

    trend = comps["trend"][series_id]
    seasonality = comps["seasonality"][series_id]
    cycle = comps["cycle"][series_id]
    regime_bias = comps["regime_bias"][series_id]
    noise = comps["noise"][series_id]
    exog_effect = comps["exog_effect"][series_id]
    reconstructed = trend + seasonality + cycle + regime_bias + noise + exog_effect

    fig, axes = plt.subplots(
        6,
        1,
        figsize=(16, 20),
        sharex=True,
        gridspec_kw={"height_ratios": [2.6, 1.5, 1.5, 1.3, 1.2, 1.1]},
    )

    axes[0].plot(time, d["y"].to_numpy(), color="#1d3557", lw=2.0, label="observed y")
    axes[0].plot(time, reconstructed, color="#e76f51", lw=1.3, alpha=0.9, label="reconstructed y")
    axes[0].set_ylabel("value")
    axes[0].set_title(f"Series {series_id}: observed signal and exact reconstruction")
    axes[0].legend(loc="upper left", ncol=2)

    axes[1].plot(time, trend, label="trend", color="#2a9d8f", lw=2.0)
    axes[1].plot(time, seasonality, label="seasonality", color="#f4a261", lw=1.8)
    axes[1].plot(time, cycle, label="cycle", color="#264653", lw=1.6)
    axes[1].plot(time, regime_bias, label="regime bias", color="#8d5a97", lw=1.4)
    axes[1].set_ylabel("component")
    axes[1].set_title("Structural components")
    axes[1].legend(loc="upper left", ncol=4)

    axes[2].plot(time, noise, label="noise", color="#457b9d", lw=1.6)
    axes[2].plot(time, exog_effect, label="exog effect", color="#d62828", lw=1.6)
    axes[2].plot(time, d["y"].to_numpy() - reconstructed, label="reconstruction error", color="#6c757d", lw=1.0, ls="--")
    axes[2].axhline(0.0, color="black", lw=0.8, alpha=0.45)
    axes[2].set_ylabel("value")
    axes[2].set_title("Stochastic and exogenous contributions")
    axes[2].legend(loc="upper left", ncol=3)

    if exog_cols:
        for col in exog_cols:
            axes[3].plot(time, d[col].to_numpy(), lw=1.5, label=col)
        axes[3].set_title("Exogenous inputs stored in the dataframe")
        axes[3].legend(loc="upper left", ncol=max(1, len(exog_cols)))
    else:
        axes[3].text(0.01, 0.5, "No exogenous features were generated.", transform=axes[3].transAxes, va="center")
        axes[3].set_title("Exogenous inputs")
    axes[3].set_ylabel("x")

    calendar_cols = ["fourier_wk_sin", "fourier_wk_cos", "fourier_yr_sin", "fourier_yr_cos"]
    for col in calendar_cols:
        axes[4].plot(time, d[col].to_numpy(), lw=1.2, label=col)
    axes[4].set_ylabel("calendar")
    axes[4].set_title("Calendar/Fourier features")
    ax_state = axes[4].twinx()
    if meta["states"] is not None:
        ax_state.step(time, meta["states"][series_id], where="mid", color="#7b2cbf", lw=1.2, alpha=0.55, label="state")
        ax_state.set_ylabel("state")
        lines_l, labels_l = axes[4].get_legend_handles_labels()
        lines_r, labels_r = ax_state.get_legend_handles_labels()
        axes[4].legend(lines_l + lines_r, labels_l + labels_r, loc="upper left", ncol=3)
    else:
        axes[4].legend(loc="upper left", ncol=4)

    summary_lines = [
        f"series={series_id} | points={len(d)} | complete={d['y'].notna().all()} | freq={params['freq']}",
        f"time span={time.min().date()} -> {time.max().date()} | y mean={d['y'].mean():.2f} | y std={d['y'].std():.2f}",
        f"trend={params['trend']}",
        f"seasonality={params['seasonality']}",
        f"cycle={params['cycle']}",
        f"noise={params['noise']}",
        f"regime={params['regime']}",
        f"exog={params['exog']}",
    ]
    axes[5].axis("off")
    axes[5].text(0.01, 0.98, "\n".join(summary_lines), va="top", ha="left", family="monospace", fontsize=10)

    axes[-1].set_xlabel("time")
    plt.tight_layout()
    plt.show()


In [ ]:
for series_id in sorted(df["series"].unique()):
    plot_full_series_info(df, meta, int(series_id))
